# Project 01: Exploratory Data Analysis (EDA) on the Titanic Dataset

**Objective:** I load the Titanic passenger data, clean it, and use pandas to find out who was most likely to survive.

## 1. Questions I want to answer
1. Did women survive more often than men?
2. Did ticket class matter?
3. Was age related to survival?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
df = sns.load_dataset("titanic")

## 2. Loading and inspecting the data
I start with the shape, the first rows, the data types and summary statistics.

In [ ]:
print("Rows, columns:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Column meanings**
- `survived`: 0 = died, 1 = survived
- `pclass` / `class`: ticket class (1 = First, 2 = Second, 3 = Third)
- `sex`, `age`: passenger's sex and age in years
- `sibsp`, `parch`: siblings/spouses and parents/children aboard
- `fare`: ticket price
- `embarked` / `embark_town`: port of boarding
- `who`, `adult_male`: man, woman or child; whether an adult man
- `deck`: cabin deck letter
- `alive`: yes/no version of `survived`
- `alone`: travelling without family aboard

## 3. Finding missing values

In [ ]:
missing = df.isna().sum()
pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(1)}).query("missing > 0")

## 4. Fixing missing values, and why
| Column | Fix | Reason |
|---|---|---|
| `age` | Median age of the same class and sex | Age differs by class and sex, so a group median is more accurate than one overall value. The median is not distorted by extreme ages. |
| `deck` | Dropped | About three quarters of the values are missing, so filling it would mostly be guessing. |
| `embarked`, `embark_town` | Most common value | Only 2 rows are missing, so the effect is negligible. |

In [ ]:
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns=["deck"])

print("Missing values left:", df.isna().sum().sum())

## 5. Duplicates and data types

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

This dataset has no passenger ID, so identical rows are most likely different people who share the same recorded details (for example, two lone third-class men of the same age). Dropping them would remove real passengers, so I **kept them**.

In [ ]:
for col in ["pclass", "sex", "embarked"]:
    df[col] = df[col].astype("category")
df.dtypes

## 6. Grouping and comparing survival

In [ ]:
overall = df["survived"].mean()
print(f"Overall survival rate: {overall:.0%}")

by_sex = df.groupby("sex", observed=True)["survived"].mean().sort_values(ascending=False)
by_sex.round(2)

In [ ]:
by_class = (df.groupby("pclass", observed=True)["survived"]
              .agg(["mean", "count"])
              .sort_values("mean", ascending=False)
              .round(2))
by_class

In [ ]:
df["age_band"] = pd.cut(df["age"], [0, 12, 18, 40, 60, 100],
                        labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"])

by_age = (df.groupby("age_band", observed=True)["survived"]
            .agg(["mean", "count"])
            .sort_values("mean", ascending=False)
            .round(2))
by_age

### Extra check: sex and class together
I want to see whether the sex gap holds inside every class.

In [ ]:
df.pivot_table(values="survived", index="pclass", columns="sex",
               aggfunc="mean", observed=True).round(2)

### Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

sns.barplot(x="sex", y="survived", data=df, ax=axes[0], errorbar=None)
axes[0].set_title("Survival rate by sex")

sns.barplot(x="pclass", y="survived", data=df, ax=axes[1], errorbar=None)
axes[1].set_title("Survival rate by class")

sns.barplot(x="age_band", y="survived", data=df, ax=axes[2], errorbar=None)
axes[2].set_title("Survival rate by age band")

for ax in axes:
    ax.set_ylabel("Survival rate")
plt.tight_layout()
plt.show()

## 7. Answers to my questions

In [ ]:
f, m = by_sex["female"], by_sex["male"]
print(f"Q1. Women survived at about {f:.0%} compared with about {m:.0%} for men.")

c1, c2, c3 = by_class.loc[1, "mean"], by_class.loc[2, "mean"], by_class.loc[3, "mean"]
print(f"Q2. Survival fell with ticket class: about {c1:.0%} in First, {c2:.0%} in Second and {c3:.0%} in Third.")

best, worst = by_age["mean"].idxmax(), by_age["mean"].idxmin()
print(f"Q3. Age was related to survival: the '{best}' band did best ({by_age.loc[best, 'mean']:.0%}) "
      f"and the '{worst}' band did worst ({by_age.loc[worst, 'mean']:.0%}).")

## 8. Summary of findings
1. **Sex:** Women were far more likely to survive than men (see Q1 above).
2. **Class:** Survival dropped steadily from First to Third class (see Q2).
3. **Age:** Children had the best chances, while survival was lower for the oldest and the middle age groups (see Q3).

**Limitations**
- These are associations, not proof of cause. Sex, class and age are linked to each other, so no single factor explains survival alone.
- About 20% of ages were estimated using class and sex medians, which can slightly smooth the age-band results.
- Some age bands (Child, Senior) have small counts, so their rates are less reliable.